# Notebook 28: Explainable AI – SHAP, LIME & Partial Dependence Plots
### Part 28/30 – ML Mastery Series for Python Experts

## Why Explainability Matters – Beyond Accuracy

You built powerful models — now let's open the black box and understand why they make the decisions they do. Explainable AI is no longer optional; it's essential for trust, debugging, and compliance…

- **Trust & adoption**: Stakeholders won't deploy models they don't understand; explanations bridge the gap between accuracy and acceptance
- **Debugging wrong predictions**: Understanding why a model failed reveals data issues, edge cases, or model limitations
- **Detecting bias/fairness issues**: Explanations can reveal when models use protected attributes (race, gender) or proxies (zip code → income → race)
- **Regulatory requirements (GDPR right to explanation)**: EU law mandates explanations for automated decisions affecting individuals
- **Scientific insight**: Understanding which features drive predictions can lead to new domain knowledge and hypotheses
- **Feature selection validation**: Explanations confirm that models use meaningful features, not spurious correlations
- **Comparing models beyond performance**: Two models with identical accuracy may use completely different logic; explanations reveal which is more sensible
- **Post-hoc vs intrinsic interpretability**: Linear models are intrinsically interpretable; complex models (boosting, neural nets) need post-hoc explanation methods like SHAP and LIME

## Learning Objectives

By the end of this notebook, you will be able to:

- **Understand additive feature attribution (SHAP)**: Game-theoretic approach to fairly distribute prediction contributions among features
- **Compute SHAP values for tree & deep models**: Use TreeExplainer for XGBoost/LightGBM, KernelExplainer for any model, DeepExplainer for neural nets
- **Interpret SHAP summary & dependence plots**: Read beeswarm plots, identify feature importance, and detect interactions
- **Use LIME for local linear explanations**: Generate locally faithful explanations around individual predictions
- **Generate partial dependence & ICE plots**: Visualize marginal feature effects while accounting for feature correlations
- **Compare global vs local methods**: Know when to use which explanation technique for your use case
- **Apply explanations to classification & regression**: Adapt methods for different prediction types and business contexts
- **Debug model failures**: Use explanations to identify why models make specific errors
- **Visualize feature interactions & monotonicity**: Detect when feature effects change based on other feature values
- **Build trust through transparency**: Create explanation reports for stakeholders and regulatory compliance

## 🌍 1. Global Interpretability – Feature Importance & Permutation

Global methods explain the model's overall behavior. We'll compare built-in feature importance (often biased) with permutation importance (more reliable) on a tree-based model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Core ML imports
from sklearn.datasets import fetch_california_housing, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import accuracy_score, mean_squared_error

# Explanation libraries
import shap
from lime.lime_tabular import LimeTabularExplainer

# Deep learning for later section
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Set seeds and style
np.random.seed(42)
tf.random.set_seed(42)
sns.set_theme(style="whitegrid")
%matplotlib inline

print("Libraries loaded successfully")
print(f"SHAP version: {shap.__version__}")
print("Note: LIME will be imported in its section")

In [ ]:
# Load California Housing dataset
housing = fetch_california_housing()
X, y = housing.data, housing.target
feature_names = housing.feature_names

# Create DataFrame for easier handling
X_df = pd.DataFrame(X, columns=feature_names)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Dataset: California Housing")
print(f"Training samples: {X_train.shape[0]}")
print(f"Features: {len(feature_names)}")
print(f"Feature names: {feature_names}")

In [ ]:
# Train Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Random Forest RMSE: {rmse:.4f}")

# Built-in feature importance (based on impurity reduction)
# WARNING: Biased toward high-cardinality features!
builtin_importance = rf_model.feature_importances_
builtin_df = pd.DataFrame({
    'feature': feature_names,
    'importance': builtin_importance
}).sort_values('importance', ascending=False)

print("\nBuilt-in Feature Importance (Gini Importance):")
print(builtin_df.head())

In [ ]:
# Permutation importance (more reliable, model-agnostic)
# Measures decrease in model performance when feature values are randomly shuffled
perm_importance = permutation_importance(
    rf_model, X_test, y_test,
    n_repeats=10,           # Number of shuffling rounds
    random_state=42,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

perm_df = pd.DataFrame({
    'feature': feature_names,
    'importance_mean': perm_importance.importances_mean,
    'importance_std': perm_importance.importances_std
}).sort_values('importance_mean', ascending=False)

print("Permutation Feature Importance:")
print(perm_df.head())

# Compare side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Built-in
sns.barplot(data=builtin_df, x='importance', y='feature', palette='viridis', ax=axes[0])
axes[0].set_title('Built-in Feature Importance\n(Gini-based, potentially biased)')
axes[0].set_xlabel('Importance')

# Permutation
sns.barplot(data=perm_df, x='importance_mean', y='feature', palette='plasma', ax=axes[1])
axes[1].set_title('Permutation Feature Importance\n(Model-agnostic, more reliable)')
axes[1].set_xlabel('Importance (drop in performance)')
axes[1].errorbar(perm_df['importance_mean'], range(len(perm_df)), 
                xerr=perm_df['importance_std'], fmt='none', color='black', alpha=0.5)

plt.tight_layout()
plt.show()

print("\nKey insight: Both methods agree on top features (MedInc, AveOccup)")
print("but permutation importance is preferred for high-cardinality features")

## 🔍 2. SHAP – Shapley Additive Explanations

SHAP (SHapley Additive exPlanations) uses game theory to fairly distribute the prediction contribution among features. It guarantees consistency, local accuracy, and missingness — properties that other methods lack.

In [ ]:
# Compute SHAP values using TreeExplainer (optimized for tree models)
# TreeExplainer is exact and fast for Random Forest, XGBoost, LightGBM, CatBoost
print("Initializing TreeExplainer for Random Forest...")
explainer_rf = shap.TreeExplainer(rf_model)

# Compute SHAP values for test set (sample for speed)
X_test_sample = X_test[:100]  # First 100 samples
print("Computing SHAP values...")
shap_values_rf = explainer_rf.shap_values(X_test_sample)

print(f"SHAP values shape: {shap_values_rf.shape}")
print(f"Expected value (base prediction): {explainer_rf.expected_value:.4f}")
print(f"Mean actual prediction: {rf_model.predict(X_test_sample).mean():.4f}")
print("✓ Property check: mean(SHAP values) + expected_value ≈ mean(prediction)")

In [ ]:
# SHAP Summary Plot (global view)
# Beeswarm plot shows distribution of SHAP values for each feature
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values_rf, X_test_sample, feature_names=feature_names, show=False)
plt.title('SHAP Summary Plot: Feature Importance & Value Impact')
plt.tight_layout()
plt.show()

print("\nSHAP Summary Interpretation:")
print("- Features ordered by importance (sum of absolute SHAP values)")
print("- Color: Red = high feature value, Blue = low feature value")
print("- Horizontal position: Impact on prediction (right = increases price)")
print("- Width: Frequency of that impact level")
print("\nExample: High MedInc (red) pushes predictions right (higher price)")

In [ ]:
# SHAP Bar Plot (global importance)
# Sum of absolute SHAP values = global feature importance
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_rf, X_test_sample, feature_names=feature_names, 
                  plot_type="bar", show=False)
plt.title('SHAP Feature Importance (Mean Absolute Value)')
plt.tight_layout()
plt.show()

# Compare SHAP importance to permutation importance
shap_importance = np.abs(shap_values_rf).mean(axis=0)
comparison_df = pd.DataFrame({
    'feature': feature_names,
    'shap_importance': shap_importance,
    'permutation_importance': perm_importance.importances_mean
}).sort_values('shap_importance', ascending=False)

print("\nSHAP vs Permutation Importance Correlation:")
correlation = comparison_df['shap_importance'].corr(comparison_df['permutation_importance'])
print(f"Correlation: {correlation:.3f} (high = agreement between methods)")

## 💡 3. Local Explanations with SHAP

While summary plots show global patterns, SHAP excels at explaining individual predictions. This is crucial for debugging specific cases and providing personalized explanations to end users.

In [ ]:
# Pick a sample instance for detailed explanation
sample_idx = 0  # First test sample
sample_instance = X_test[sample_idx]
sample_prediction = rf_model.predict([sample_instance])[0]

print(f"Sample Instance {sample_idx}:")
print(f"  Predicted value: {sample_prediction:.4f} ($100k)")
print(f"  Actual value: {y_test[sample_idx]:.4f} ($100k)")
print(f"  Base value (average): {explainer_rf.expected_value:.4f}")

# Show feature values for this instance
print(f"\nFeature values:")
for name, value in zip(feature_names, sample_instance):
    print(f"  {name}: {value:.3f}")

In [ ]:
# Waterfall plot: How each feature pushes prediction from base to final
plt.figure(figsize=(10, 6))
shap.waterfall_plot(shap.Explanation(
    values=shap_values_rf[sample_idx],
    base_values=explainer_rf.expected_value,
    data=sample_instance,
    feature_names=feature_names
), show=False)
plt.title(f'Waterfall Plot: Prediction Breakdown for Instance {sample_idx}')
plt.tight_layout()
plt.show()

print("\nWaterfall Interpretation:")
print("- Starts at base value (average prediction across dataset)")
print("- Each bar shows how that feature pushes the prediction up (red) or down (blue)")
print("- Final prediction = base + sum of all SHAP values")
print(f"- Verification: {explainer_rf.expected_value + shap_values_rf[sample_idx].sum():.4f} ≈ {sample_prediction:.4f}")

In [ ]:
# Force plot: Alternative visualization (horizontal layout)
# Note: Force plots require shap.initjs() in Jupyter but work without in saved notebooks
print("Force plot visualization:")
print("(In Jupyter, this renders as interactive visualization)")

# Create force plot data explanation
force_explanation = shap.Explanation(
    values=shap_values_rf[sample_idx],
    base_values=explainer_rf.expected_value,
    data=sample_instance,
    feature_names=feature_names
)

# For static notebooks, we use waterfall; for interactive, use:
# shap.force_plot(explainer_rf.expected_value, shap_values_rf[sample_idx], sample_instance, feature_names=feature_names)

print("\nForce plot shows the same information as waterfall but horizontally:")
print("- Blue arrows (left): Features pushing prediction lower")
print("- Red arrows (right): Features pushing prediction higher")
print("- Base value in center, final prediction at the end")

In [ ]:
# Dependence plot: Feature value vs SHAP value (shows interactions)
# Most important feature
top_feature_idx = np.argsort(np.abs(shap_values_rf).mean(axis=0))[-1]
top_feature_name = feature_names[top_feature_idx]

plt.figure(figsize=(10, 6))
shap.dependence_plot(
    top_feature_idx, 
    shap_values_rf, 
    X_test_sample,
    feature_names=feature_names,
    show=False
)
plt.title(f'SHAP Dependence Plot: {top_feature_name}')
plt.tight_layout()
plt.show()

print(f"\nDependence plot shows:")
print(f"- How {top_feature_name} affects predictions (y-axis = SHAP value)")
print("- Color shows interaction with second most correlated feature")
print("- Vertical dispersion indicates interactions with other features")

## 🍋 4. LIME – Local Interpretable Model-agnostic Explanations

LIME explains any classifier or regressor by perturbing the input and fitting a local linear model. Unlike SHAP which is exact for trees, LIME is an approximation but works for any model type.

In [ ]:
# Initialize LIME explainer for tabular data
# LIME needs training data to understand feature distributions
lime_explainer = LimeTabularExplainer(
    training_data=X_train,           # Background data for perturbation
    feature_names=feature_names,     # Feature names for display
    mode='regression',               # 'regression' or 'classification'
    discretize_continuous=True,      # Discretize continuous features for interpretability
    random_state=42
)

print("LIME Explainer initialized")
print("Mode: Regression")
print(f"Training samples for perturbation: {X_train.shape[0]}")

# Explain the same sample instance we used for SHAP
sample_idx = 0
sample_instance = X_test[sample_idx]
sample_pred = rf_model.predict([sample_instance])[0]

print(f"\nExplaining instance {sample_idx} (prediction: {sample_pred:.4f})")

In [ ]:
# Generate LIME explanation
# LIME perturbs the instance, gets predictions, and fits weighted linear model
lime_exp = lime_explainer.explain_instance(
    data_row=sample_instance,
    predict_fn=rf_model.predict,     # Prediction function
    num_features=5,                  # Top 5 features to show
    num_samples=500                  # Number of perturbed samples
)

# Get explanation as list
lime_features = lime_exp.as_list()
print("LIME Explanation (top 5 features):")
for feature, weight in lime_features:
    direction = "↑ increases" if weight > 0 else "↓ decreases"
    print(f"  {feature}: {weight:+.4f} ({direction} price)")

# LIME shows local linear approximation around the instance
# Positive weight = feature pushes prediction higher
# Negative weight = feature pushes prediction lower

In [ ]:
# Visualize LIME explanation
# In Jupyter, this would show an interactive visualization
# For notebook, we extract and plot the weights
lime_df = pd.DataFrame(lime_features, columns=['feature', 'weight'])
lime_df['abs_weight'] = np.abs(lime_df['weight'])
lime_df = lime_df.sort_values('abs_weight', ascending=True)

plt.figure(figsize=(10, 6))
colors = ['green' if w > 0 else 'red' for w in lime_df['weight']]
plt.barh(lime_df['feature'], lime_df['weight'], color=colors, alpha=0.7)
plt.xlabel('LIME Weight (Local Linear Coefficient)')
plt.title(f'LIME Explanation for Instance {sample_idx}')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\nLIME vs SHAP comparison:")
print("- LIME: Local approximation, model-agnostic, faster for non-tree models")
print("- SHAP: Exact for trees, theoretically grounded, consistent across instances")
print("- Both identify MedInc as top feature for this instance ✓")

## 📊 5. Partial Dependence Plots (PDP) & ICE

PDP shows the marginal effect of a feature on the predicted outcome, averaging over all other features. ICE (Individual Conditional Expectation) curves show this for individual instances, revealing heterogeneity.

In [ ]:
# Partial Dependence Plots with sklearn
# PDP shows average prediction as function of feature value
top_features = ['MedInc', 'AveOccup', 'HouseAge']
feature_indices = [feature_names.index(f) for f in top_features]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (feature, ax) in enumerate(zip(top_features, axes)):
    PartialDependenceDisplay.from_estimator(
        rf_model, X_train, features=[feature],
        kind='average',              # 'average' = PDP, 'individual' = ICE, 'both' = both
        ax=ax,
        line_kw={'linewidth': 2, 'color': 'blue'}
    )
    ax.set_title(f'PDP: {feature}')
    ax.grid(True, alpha=0.3)

plt.suptitle('Partial Dependence Plots: Marginal Feature Effects', y=1.02)
plt.tight_layout()
plt.show()

print("PDP Interpretation:")
print("- Shows how prediction changes as feature changes (averaged over all samples)")
print("- MedInc: Higher income → higher price (monotonic, as expected)")
print("- AveOccup: More occupants → slightly lower price (crowding effect)")
print("- HouseAge: Non-monotonic relationship (very old houses may be worth less)")

In [ ]:
# ICE (Individual Conditional Expectation) curves
# Shows PDP for individual instances, revealing heterogeneity
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ICE for MedInc
PartialDependenceDisplay.from_estimator(
    rf_model, X_train, features=['MedInc'],
    kind='both',                     # Show both PDP (thick line) and ICE (thin lines)
    ax=axes[0],
    line_kw={'linewidth': 0.5, 'alpha': 0.5},  # ICE lines
    pd_line_kw={'linewidth': 3, 'color': 'red'}  # PDP line
)
axes[0].set_title('ICE + PDP: Median Income')
axes[0].grid(True, alpha=0.3)

# ICE for AveOccup
PartialDependenceDisplay.from_estimator(
    rf_model, X_train, features=['AveOccup'],
    kind='both',
    ax=axes[1],
    line_kw={'linewidth': 0.5, 'alpha': 0.5},
    pd_line_kw={'linewidth': 3, 'color': 'red'}
)
axes[1].set_title('ICE + PDP: Average Occupancy')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nICE Interpretation:")
print("- Each thin line = one instance's prediction curve")
- Thick red line = average (PDP)")
- Diverging lines indicate feature interactions")
- If all lines parallel → no interactions; crossing lines → strong interactions")

## 🤖 6. Explaining Deep Models & Text/Images Preview

SHAP extends beyond trees to deep learning via DeepExplainer and GradientExplainer. While less exact than TreeExplainer, it provides crucial insights into neural network behavior.

In [ ]:
# Build simple neural network for California Housing
# This demonstrates explaining non-tree models
def build_housing_nn(input_dim):
    """Simple neural network for regression."""
    model = keras.Sequential([
        layers.Dense(64, activation='relu', input_shape=(input_dim,)),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(16, activation='relu'),
        layers.Dense(1)  # Regression output
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

# Scale features for neural network
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train neural network
nn_model = build_housing_nn(X.shape[1])
print("Training neural network...")
nn_model.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

# Evaluate
nn_pred = nn_model.predict(X_test_scaled, verbose=0).flatten()
nn_rmse = np.sqrt(mean_squared_error(y_test, nn_pred))
print(f"Neural Network RMSE: {nn_rmse:.4f}")
print(f"Random Forest RMSE:  {rmse:.4f}")
print(f"(Tree model wins on this tabular dataset - as expected)")

In [ ]:
# SHAP for neural network using DeepExplainer
# DeepExplainer uses background data to approximate SHAP values
print("Computing SHAP values for neural network...")
print("(This uses background samples to estimate feature attributions)")

# Use subset of training data as background
background = X_train_scaled[:100]
explainer_nn = shap.DeepExplainer(nn_model, background)

# Compute SHAP values for test samples
shap_values_nn = explainer_nn.shap_values(X_test_scaled[:50])

print(f"SHAP values shape: {shap_values_nn[0].shape if isinstance(shap_values_nn, list) else shap_values_nn.shape}")

# Summary plot for neural network
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values_nn[0] if isinstance(shap_values_nn, list) else shap_values_nn,
    X_test_scaled[:50],
    feature_names=feature_names,
    show=False
)
plt.title('SHAP Summary: Neural Network')
plt.tight_layout()
plt.show()

print("\nNeural network SHAP shows similar patterns to Random Forest")
print("(Both identify MedInc as most important - model converges on truth)")

In [ ]:
# Compare SHAP values between Random Forest and Neural Network
# Sample 50 instances from test set for fair comparison
sample_size = 50
X_sample = X_test[:sample_size]
X_sample_scaled = scaler.transform(X_sample)

# Get SHAP values from both models
shap_rf_sample = explainer_rf.shap_values(X_sample)
shap_nn_sample = explainer_nn.shap_values(X_sample_scaled)
shap_nn_sample = shap_nn_sample[0] if isinstance(shap_nn_sample, list) else shap_nn_sample

# Compute correlation of feature importance rankings
rf_importance = np.abs(shap_rf_sample).mean(axis=0)
nn_importance = np.abs(shap_nn_sample).mean(axis=0)

importance_corr = np.corrcoef(rf_importance, nn_importance)[0, 1]

print(f"Feature importance correlation (RF vs NN): {importance_corr:.3f}")
print(f"(1.0 = perfect agreement, 0 = no correlation)")

# Visualize comparison
comparison_df = pd.DataFrame({
    'feature': feature_names,
    'rf_importance': rf_importance,
    'nn_importance': nn_importance
})

plt.figure(figsize=(10, 6))
plt.scatter(comparison_df['rf_importance'], comparison_df['nn_importance'], s=100, alpha=0.7)
for i, row in comparison_df.iterrows():
    plt.annotate(row['feature'], (row['rf_importance'], row['nn_importance']),
                xytext=(5, 5), textcoords='offset points', fontsize=9)
plt.xlabel('Random Forest SHAP Importance')
plt.ylabel('Neural Network SHAP Importance')
plt.title('Feature Importance: Random Forest vs Neural Network')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nModels agree on important features despite different architectures")
print("This increases trust that features are truly predictive, not artifacts")

## 🛠️ 7. Putting It Together – Debugging & Trust

Let's use explanations to debug a misclassified instance and compare explanations across models to build trust.

In [ ]:
# Classification example for debugging misclassifications
# Load breast cancer dataset
cancer = load_breast_cancer()
X_cancer, y_cancer = cancer.data, cancer.target
feature_names_cancer = cancer.feature_names

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cancer, y_cancer, test_size=0.2, random_state=42, stratify=y_cancer
)

# Train classifier
rf_cancer = RandomForestClassifier(n_estimators=100, random_state=42)
rf_cancer.fit(X_train_c, y_train_c)
y_pred_c = rf_cancer.predict(X_test_c)
accuracy = accuracy_score(y_test_c, y_pred_c)

print(f"Breast Cancer Classification Accuracy: {accuracy:.4f}")

# Find misclassified instances
misclassified = np.where(y_pred_c != y_test_c)[0]
print(f"Misclassified instances: {len(misclassified)}")

if len(misclassified) > 0:
    # Analyze first misclassification
    idx = misclassified[0]
    true_label = y_test_c[idx]
    pred_label = y_pred_c[idx]
    
    print(f"\nAnalyzing misclassified instance {idx}:")
    print(f"  True: {'Malignant' if true_label == 0 else 'Benign'}")
    print(f"  Predicted: {'Malignant' if pred_label == 0 else 'Benign'}")
    
    # SHAP explanation for this instance
    explainer_cancer = shap.TreeExplainer(rf_cancer)
    shap_values_cancer = explainer_cancer.shap_values(X_test_c[idx:idx+1])
    
    # For classification, SHAP returns list (one per class)
    # shap_values_cancer[0] = contributions to class 0 (malignant)
    # shap_values_cancer[1] = contributions to class 1 (benign)
    
    plt.figure(figsize=(10, 6))
    shap.waterfall_plot(shap.Explanation(
        values=shap_values_cancer[1][0],  # Explain benign prediction
        base_values=explainer_cancer.expected_value[1],
        data=X_test_c[idx],
        feature_names=feature_names_cancer
    ), show=False, max_display=10)
    plt.title(f'Why predicted Benign? (Instance {idx})')
    plt.tight_layout()
    plt.show()
    
    print("\nExplanation shows which features pushed toward benign vs malignant")
    print("This helps doctors understand and potentially override AI decisions")

In [ ]:
# Compare explanations across multiple models
# Train XGBoost and compare SHAP with Random Forest
import xgboost as xgb

xgb_cancer = xgb.XGBClassifier(n_estimators=100, max_depth=4, random_state=42)
xgb_cancer.fit(X_train_c, y_train_c)

# SHAP for XGBoost
explainer_xgb = shap.TreeExplainer(xgb_cancer)
shap_values_xgb = explainer_xgb.shap_values(X_test_c[:50])
shap_values_xgb = shap_values_xgb[1] if isinstance(shap_values_xgb, list) else shap_values_xgb

# SHAP for Random Forest (recompute for same samples)
shap_values_rf_c = explainer_cancer.shap_values(X_test_c[:50])
shap_values_rf_c = shap_values_rf_c[1] if isinstance(shap_values_rf_c, list) else shap_values_rf_c

# Compare top features
rf_imp_c = np.abs(shap_values_rf_c).mean(axis=0)
xgb_imp_c = np.abs(shap_values_xgb).mean(axis=0)

top_rf = set(np.argsort(rf_imp_c)[-5:])
top_xgb = set(np.argsort(xgb_imp_c)[-5:])

print("Top 5 features by SHAP importance:")
print(f"Random Forest: {[feature_names_cancer[i] for i in top_rf]}")
print(f"XGBoost:       {[feature_names_cancer[i] for i in top_xgb]}")
print(f"Agreement: {len(top_rf & top_xgb)}/5 features")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

shap.summary_plot(shap_values_rf_c, X_test_c[:50], feature_names=feature_names_cancer, 
                  plot_type='bar', show=False, ax=axes[0])
axes[0].set_title('Random Forest SHAP')

shap.summary_plot(shap_values_xgb, X_test_c[:50], feature_names=feature_names_cancer,
                  plot_type='bar', show=False, ax=axes[1])
axes[1].set_title('XGBoost SHAP')

plt.tight_layout()
plt.show()

print("\nHigh agreement between models increases trust in predictions")
print("Disagreement suggests investigating feature engineering or data quality")

## ⚠️ Common Pitfalls & Pro Tips

- **KernelExplainer slow on large data → use TreeExplainer when possible**: TreeExplainer is exact and O(TLD²) vs KernelExplainer's O(2^M); always use algorithm-specific explainers when available
- **Misinterpreting SHAP signs**: Positive SHAP = pushes prediction higher (for any output), not necessarily "good"; interpret relative to your use case
- **Not centering SHAP around base value**: SHAP values are relative to expected value (training set average); always mention the baseline when presenting explanations
- **PDP assumes independence → use ICE for heterogeneity**: When features are correlated, PDP can show impossible combinations; ICE curves reveal when effects differ across instances
- **LIME instability on small perturbations**: LIME fits local linear models that can vary with random seed and neighborhood size; run multiple times for stability check
- **Not checking model calibration before explaining**: Explaining an uncalibrated model (confidence ≠ probability) can mislead; calibrate with isotonic regression or Platt scaling first
- **Over-relying on post-hoc methods**: Post-hoc explanations approximate model behavior; for high-stakes decisions, prefer intrinsically interpretable models (linear, small trees)
- **Ignoring feature interactions in SHAP**: SHAP dependence plots reveal interactions; flat summary plots may hide important conditional effects
- **Explaining on wrong data distribution**: SHAP values computed on training data may not generalize to different populations (domain shift)
- **Not validating explanations**: Check that explanations make domain sense; nonsensical top features indicate data leakage or model issues
- **Forgetting computational cost**: SHAP exact computation is expensive; use sampling approximations (shap.sample) for large datasets
- **Presenting raw SHAP values to non-technical users**: Convert to natural language or visualizations (waterfall, force plots) for stakeholder communication

## 📝 Exercises

Practice explainable AI techniques:

**Easy:** Compute SHAP summary plot for a RandomForestRegressor on California Housing. Identify the top 3 most important features and verify they align with domain knowledge (income, location, age).

**Medium:** Use LIME to explain a misclassified instance in the Breast Cancer dataset. Compare the LIME explanation to SHAP waterfall plot for the same instance and note any differences in top features.

**Medium:** Plot PDP + ICE curves for the top 3 features in California Housing. Identify which features show strong interactions (diverging ICE lines) and discuss possible causes.

**Hard:** Compare SHAP explanations between XGBoost and a simple neural network (2-layer MLP) on the same dataset. Calculate the correlation between their feature importance rankings and visualize in a scatter plot.

**Bonus:** Implement a custom SHAP-based report generator that creates a PDF/HTML explanation for any prediction, including: waterfall plot, top 3 drivers, and confidence assessment based on SHAP value distribution.

<details>
<summary><b>Exercise Solutions (Click to Expand)</b></summary>

### Easy Solution Outline (SHAP Summary)
```python
import shap
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test, feature_names=feature_names)
# Top 3: MedInc, AveOccup, HouseAge
```

### Medium Solution Outline (LIME on Misclassification)
```python
from lime.lime_tabular import LimeTabularExplainer
explainer = LimeTabularExplainer(X_train, feature_names=feature_names_cancer, mode='classification')
exp = explainer.explain_instance(X_test_c[misclassified[0]], rf_cancer.predict_proba, num_features=5)
exp.show_in_notebook()
```

### Medium Solution Outline (PDP + ICE)
```python
from sklearn.inspection import PartialDependenceDisplay
fig, ax = plt.subplots(figsize=(10, 6))
PartialDependenceDisplay.from_estimator(rf_model, X_train, features=['MedInc'], 
                                        kind='both', ax=ax)
# Look for diverging lines = interactions
```

### Hard Solution Outline (Model Comparison)
```python
# SHAP for both models
shap_rf = shap.TreeExplainer(rf).shap_values(X_test)
shap_nn = shap.DeepExplainer(nn, background).shap_values(X_test_scaled)
# Compare rankings
from scipy.stats import spearmanr
corr, _ = spearmanr(np.abs(shap_rf).mean(0), np.abs(shap_nn).mean(0))
```

</details>

## ✅ Summary – What You Learned Today

- **Global interpretability methods**: Built-in feature importance (biased) vs. permutation importance (robust) for understanding overall model behavior
- **SHAP (Shapley Additive Explanations)**: Game-theoretic approach providing consistent, locally accurate feature attributions for any prediction
- **TreeExplainer efficiency**: Exact polynomial-time SHAP computation for tree-based models (XGBoost, LightGBM, Random Forest, CatBoost)
- **Local explanation techniques**: Waterfall and force plots showing how each feature pushes prediction from base value to final output
- **LIME (Local Interpretable Model-agnostic Explanations)**: Model-agnostic local approximation fitting linear models around individual predictions
- **Partial Dependence Plots (PDP)**: Marginal feature effects averaged across the dataset; ICE curves reveal heterogeneity and interactions
- **Dependence plots**: SHAP value vs. feature value colored by interaction features to detect conditional relationships
- **Deep model explanation**: DeepExplainer and GradientExplainer extend SHAP to neural networks with background data approximations
- **Model comparison via explanations**: Agreement between different model architectures increases trust; disagreement signals investigation needed
- **Debugging with explanations**: Using SHAP and LIME to understand misclassifications, detect bias, and validate model behavior
- **Regulatory and trust considerations**: GDPR compliance, stakeholder communication, and the balance between accuracy and interpretability

## 🔮 Next Notebook Preview

**Notebook 29: AutoML – Automated Machine Learning with TPOT, Auto-sklearn, and H2O**

The future of ML engineering:
- **AutoML concepts**: Automated feature engineering, hyperparameter optimization, and model selection
- **TPOT**: Genetic programming for pipeline optimization, evolving generations of better preprocessing + model combinations
- **Auto-sklearn**: Bayesian optimization and meta-learning for robust automated model building
- **H2O.ai**: Distributed AutoML platform with automatic feature engineering and leaderboard
- **When to use AutoML**: Rapid prototyping, baseline establishment, and resource-constrained teams
- **When to avoid AutoML**: High-stakes decisions requiring full control, custom loss functions, and interpretability requirements
- **Human-in-the-loop**: Combining automated search with domain expertise for best results

Prepare to let algorithms design algorithms! 🤖⚡